# 🔍 Notebook 02 — RAG Pipeline Demo

This notebook walks through the **Retrieval-Augmented Generation (RAG)** pipeline:

```
transcript → chunks → embeddings → FAISS → retrieve relevant context → feed to LLM
```

**Why RAG?**
Without RAG, the LLM generates summaries and quizzes from memory — not from your actual lecture.
This causes *hallucination*: the model confidently states things the lecture never covered.

With RAG, every LLM call is grounded in retrieved chunks from the real transcript.

In [ ]:
import sys, os
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')

from backend.core.chunker import chunk_text
from backend.core.embedder import LectureVectorStore

## Step 1 — Load a transcript

In [ ]:
# Load the sample transcript
with open('../examples/sample_transcript.txt', 'r') as f:
    transcript = f.read()

print(f"Transcript loaded: {len(transcript):,} characters, {len(transcript.split()):,} words")

## Step 2 — Chunk the transcript

We split the transcript into overlapping 800-character chunks.
Overlap (100 chars) ensures concepts that span a boundary aren't cut off.

In [ ]:
chunks = chunk_text(transcript)

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")
print()
print("=== Sample chunk (index 0) ===")
print(chunks[0])
print()
print("=== Sample chunk (index 1) ===")
print(chunks[1][:200], '...')

## Step 3 — Build the vector store

Each chunk is converted to a 1536-dimensional embedding vector using OpenAI's
`text-embedding-3-small` model, then stored in a FAISS index for fast similarity search.

⚠️ This requires your `OPENAI_API_KEY` to be set in `.env`

In [ ]:
store = LectureVectorStore()
store.build(chunks)
print("✅ Vector store built")

## Step 4 — Retrieve relevant chunks

This is the core of RAG. We query the vector store with natural language
and get back the most semantically similar chunks — ready to pass to the LLM.

In [ ]:
# Try different queries and see what comes back
queries = [
    "main topic and key concepts",
    "technical terms and definitions",
    "practical examples and applications",
]

for query in queries:
    print(f"Query: '{query}'")
    context = store.retrieve(query, k=2)
    print(context[:300])
    print('---')

## Step 5 — Compare: With RAG vs Without RAG

This cell demonstrates *why RAG matters* by generating a summary
with and without retrieved context.

In [ ]:
from openai import OpenAI
client = OpenAI()

# WITHOUT RAG — no context, just the topic name
response_no_rag = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Summarise a lecture about machine learning in 3 sentences."}],
    max_tokens=200, temperature=0.3
)

# WITH RAG — grounded in actual transcript content
context = store.retrieve("main topic overview key points", k=5)
response_rag = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": f"Based ONLY on this lecture content, summarise in 3 sentences:\n\n{context}"}],
    max_tokens=200, temperature=0.3
)

print("WITHOUT RAG (generic, may not match the lecture):")
print(response_no_rag.choices[0].message.content)
print()
print("WITH RAG (grounded in actual transcript):")
print(response_rag.choices[0].message.content)